In [ ]:
#hide
#default_exp tonal_algebra

from nbdev.showdoc import *
import music21 as m
import itertools

D_LEN = 7  # "Diatonic Length" - The number of tones in a diatonic scale.
C_LEN = 12 # "Chromatic Length" - The number of tones in a chromatic scale.

In [ ]:
%%html
<style>
.output_stderr {
    display: none;
}
</style>

# Tonal Algebra

> An implementation of a novel, non-lossy, and mathematically consistent vectorization method for musical pitch.

**ABSTRACT:** Current methods for encoding musical pitch information are either lossy or not mathematically consistent and useful. This module explains why that is, demonstrates an novel, vector-based approach to music pitch encoding which is non-lossy as well as mathematically useful and consistent, and implements the core algorithms of this method using Python primitive types (tuples and integers) and simple arithmetic operations.

**Note:** This approach is only intended to encode musical pitch and pitch-related information (letter names, intervals, intervallic quality, transpositions) as these concepts are understood in the Western Classical and Popular music theory and notation systems. While it may be possible to adapt the core concepts to other musical systems, such work is currently outside both the scope of this project and the expertise of its author.

## Current problems in pitch encoding

The majority of pitch encoding schemas assume an equal-tempered, twelve-tone scale in which members of so-called "enharmonic" pitch sets (such as {G$\flat$, F$\sharp$} ) are treated as if they were the same note.

The typical encoding is:

> 0  = C (and B$\sharp$)  
> 1  = D$\flat$, C$\sharp$  
> 2  = D  
> 3  = E$\flat$, D$\sharp$  
> 4  = E (and F$\flat$)  
> 5  = F (and E$\sharp$)  
> 6  = G$\flat$, F$\sharp$  
> 7  = G  
> 8  = A$\flat$, G$\sharp$  
> 9  = A  
> 10 = B$\flat$, A$\sharp$  
> 11 = B (and C$\flat$)

The problems with such an encoding should be obvious to a reasonably proficient musician.

### Chordal analysis problems

The particular quality and function of a harmonic structure (a chord, for example) is based on the quality of the intervals from the root. For example, in a simple C Major triad, the quality of the third (E$\natural$ or E$\flat$) and the fifth (G$\natural$, G$\sharp$, G$\flat$). One can show the result of various combination in a simple chart:

```
C       Gb     G      G#

Eb      Cdim   Cmin   N/A

E       N/A    Cmaj   Caug
```

The chord {C, E$\flat$, G$\flat$} is *obviously* a C diminished triad.
But how should one interpret any of the following?

- {C, D$\sharp$, G$\flat$} - Cdim5$\sharp$9 ?
- {C, E$\flat$, F$\sharp$} - Cmin Add$\sharp$4 ?
- {C, D$\sharp$, F$\sharp$} - D$\sharp$dim$\flat$7/C (third inversion with an omitted fifth) ?

These analyses don't make sense because the letter names communicate the wrong information; three halfsteps up from the root of a chord is either minor third or it is an augmented second. And which one it is matters to how a chord is understood.

Not incidentally, one of the "N/A" (not applicable) chords present an interesting example as well.

{C, E$\flat$, G$\sharp$} looks like some kind of hard-to-analyse C-root triad. But respelling {0, 2, 8} *properly* yields it's more obvious analysis. {G$\sharp$, B$\sharp$, D$\sharp$} is a simple G$\sharp$ major triad.

Unless it is the even simpler A$\flat$ major triad.

(Making sense of {C, E, G$\flat$} is left as an exercise for the reader.)

### Progression analysis problems

This is simply an extension of the problem addressed above in chordal analysis.

Given an integer-encoded note set {1, 3, 7, 10}, how should this chord be analysed?

Setting aside the difficulty in determinging which of those is the root of the chord (4), what set of rules would definitively choose between two options:

- D$\sharp$7
- E$\flat$7

Consider the following two progressions:

> {1, 3, 7, 10} -> {0, 3, 6, 8}  
> {1, 3, 7, 10} -> {1, 4, 7, 11}

Showing what each scalar value could represent, we have:

> {{D$\flat$, C$\sharp$}, {E$\flat$, D$\sharp$}, {G}, {B$\flat$, A$\sharp$}} -> {{C}, {E$\flat$, D$\sharp$}, {G$\flat$, F$\sharp$}, {A$\flat$, G$\sharp$}}  
> {{D$\flat$, C$\sharp$}, {E$\flat$, D$\sharp$}, {G}, {B$\flat$, A$\sharp$}} -> {{D$\flat$, C$\sharp$}, {E}, {G}, {B}}

Can you write an algorithm that successfully analyses these two progressions? Could you even write an algorithm that would successfully determine which member of each enharmonic pair is the "correct" one?

### Performance problems

#### Sightreading and improvisation

When a performer sees {D, F$\sharp$, A, C$\sharp$) in a score, Dmaj7 is easily understood. They know where to place their hands on the instrument, what additional notes or scales can be played, and how the chord relates to the key of the piece as a whole.

On the other hand {D, G$\flat$, A, D$\flat$} is simply confusing. It is not immediately clear what is intended. It is a mistake.

#### Continuous and non-tempered tuning

Piano keyboards treat G$\flat$ and F$\sharp$ as the same note because they have equalized (or "tempered") the tuning across twelve notes in an octave. This is neccessary in order to have finite and reasonable number of fixed, discreet pitches. However, many instruments (notably the viols and the human voice) do not rely on quantized intervals but can tune continuously across the available pitch space. To a violin player, G$\flat$ and F$\sharp$ *really are* different notes, which will be tuned differently. They are not simply two different ways to write the same note.

This is, ultimately, the primary problem...

### Ontological problem

C$\sharp$ and D$\flat$ are different notes.

A key on a piano (or MIDI note ID) is one particular representation of a note, but it isn't the note. An encoding system that represents piano keys is now two steps removed from the essence of a note, a representation of a representation.

**The goal of a musical encoding should be to represent music in a way that preserves as much semantic meaning as possible. You cannot teach a computer how to reason about music if you give it less information than a first year music student would get.**

## Tonal Vector: musical pitches as two-tuples

The identification of pitchclass name can be accomplished with two pieces of information:

- it's "diatonic" value (that is, what letter name it has)
- it's "chromatic" value (that is, what keyboard key it is)

Every pitchclass name can then be translated into a tuple of the form `(d, c)`.
This form shall be called a `TonalVector`.

Note: Besides *diatonic* and *chromatic*, you may also think of these as *discrete* and *continuous*. The present implementation assumes `c` is an integer. However, a decimal number could be used to represent microtonal pitches or just intonation.

![keyboard](images/keyboard.png)

Any pitchclass name based on **C** (from C-doubleflat to C-doublesharp) would have a `d` value of `0`, but they would be distinguished by their `c` value:

- C$\flat$ = `(0, 11)`
- C$\natural$ = `(0,  0)`
- C$\sharp$ = `(0,  1)`

### Intervallic vectors

Each of these two-tuples can also be used to identify the abstract interval, as measured from an origin of `(0, 0)`. 

For example:

- E = `(2, 4)`
- E is a Major Third up from C.
- Therefore `(2, 4)` also represents any major third.

This can be used as a vector for addition and subtraction:

What is a Major Third up from E$\flat$? 

- Major Third = `(2, 4)`
- E$\flat$ = `(2, 3)`
- `(2, 3) + (2, 4) = (4, 7)`
- `(4, 7)` = G

What is a Minor Second down from F?

- Minor Second = `(1, 1)`
- F = `(3, 5)`
- `(3, 5) - (1, 1) = (2, 4)`
- `(2, 4)` = E

What is the interval between D and A$\flat$?

- A$\flat$ = `(5, 8)`
- D = `(1, 2)`
- `(5, 8) - (1, 2) = (4, 6)`
- `(4, 6)` = Diminished Fifth

#### Modulo

Obviously, intervals can be built which cross the B/C octave divide. The implementation below takes this into account by using modulo arithmetic to keep the result of operations within the `(0:6)` and `(0:11)` ranges. Special care must be taken to account for C$\flat$ and B$\sharp$.

#### Octave

A third scalar can be added to the Tonal Vector, qualifying its octave. This system assigns Middle C to `(0, 0, 0)`.


In [ ]:
#hide

def _tonal_modulo(x):
    """Returns an octave-normalized rendering of x.
    Examples
    --------
    >>> _tonal_modulo((7, 12)) # C + 1 octave, no octave designation
    (0, 0)
    >>> _tonal_modulo((7, 12, 0)) # C + 1 octave
    (0, 0, 1)
    >>> _tonal_modulo((-1, -1)) # B - 1 octave
    (6, 11)
    >>> _tonal_modulo((-1, -1, 0)) # B - 1 octave
    (6, 11, -1)
    >>> _tonal_modulo((-1, 0))
    (6, 0)
    >>> _tonal_modulo((7, 12, 1))
    (0, 0, 2)
    """

    # From (0,0) to (6,11) (inclusive), no modulo is needed.
    if x[0] in range(D_LEN) and x[1] in range(C_LEN):
        return x

    d_val = x[0] % D_LEN # The normalized diatonic value.
    d_oct = x[0] // D_LEN # The additional diatonic octave.
    c_val = x[1] % C_LEN # The normalized chromatic value.
    
    if len(x) == 2:
        return (d_val, c_val)

    if len(x) == 3:
        return (d_val, c_val, (x[2] + d_oct))

## Operations on Tonal Vectors

While the `TonalVector` class provides a "smart" representation of the Tonal Vector, including arithmetic operations, translations from conventional pitch names to vectors, and other useful features, it is possible to represent the Tonal Vector idea as a raw vector of scalars. Therefore operations on Tonal Vectors are implemented here as pure functions.

In [ ]:
#export

def tonal_sum(x, y):
    """Returns the value of x + y.
    
    Examples
    --------
    >>> tonal_sum((0, 0), (2, 3))
    (2, 3)
    >>> tonal_sum((3, 6), (4, 6))
    (0, 0)
    >>> tonal_sum((0, 0, 0), (2, 3))
    (2, 3, 0)
    >>> tonal_sum((3, 6, 0), (4, 6))
    (0, 0, 1)
    >>> tonal_sum((6, 11, 1), (2, 4))
    (1, 3, 2)
    """

    if len(x) < len(y):
        raise TypeError("An octave designation cannot be added to an abstract tonal value.")

    sum = tuple(xval+yval for xval,yval in itertools.zip_longest(x,y, fillvalue=0))

    sum = _tonal_modulo(sum)

    return sum

In [ ]:
D = (1, 2)
min3 = (2, 3)
tonal_sum(D, min3) # (3, 5) = F


(3, 5)

In [ ]:
fsharp = (3, 6, 0) # Above middle C
dim5 = (4, 6) # Diminished fifth, no octave qualification

tonal_sum(fsharp, dim5) # (0, 0, 1) C above middle C

(0, 0, 1)